<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>HW2 @ Deep Learning Course, Dr. Soleymani</b></p>
    <p><b>Desinged by Shaygan Adim</b></p>
</div>

---




*Full Name: Shayan Salehi*

*Student Number: 404203509*

## Homework Overview
In this homework, you will learn to implement, train, and evaluate Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) models on a text classification task using a dataset of IMDB movie reviews, and compare them.

**NOTE : Be sure to answer the analytical questions at the end of the notebook as well.**

In [1]:
# Imports
import nltk
nltk.download('stopwords')
import os
import random
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk import wordpunct_tokenize
from tqdm import tqdm
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from IPython.core.display import display, HTML
from datasets import load_dataset

# Enable tqdm progress bar in pandas
tqdm.pandas()

# Set device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Using device: cuda


# Dataset

In this section, we’ll load the IMDB dataset and preprocess the data to make it suitable for training RNN and LSTM models.

## Load Dataset
Description of Dataset: The IMDB movie reviews dataset consists of reviews along with their labels (positive or negative sentiment). Each review is a sentence or paragraph of text.

Download the Dataset: We will use a Huggingface to download the dataset into our environment.

In [2]:
# Load the IMDb dataset (Hugging Face)
dataset = load_dataset("imdb")

# Make sure 'data' folder exists
os.makedirs('data', exist_ok=True)

# Combine train and test splits into one DataFrame
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Save combined DataFrame to CSV
DATA_PATH = 'data/imdb_reviews.csv'
full_df.to_csv(DATA_PATH, index=False)

print(f"✅ Saved combined IMDb reviews to: {DATA_PATH}")
print(f"Total samples: {len(full_df)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

✅ Saved combined IMDb reviews to: data/imdb_reviews.csv
Total samples: 50000


In [3]:
# Show 5 random samples from the DataFrame
print(full_df.sample(5))

                                                    text  label
12625  Bromwell High is nothing short of brilliant. E...      1
10474  Director Kevin Connor and wannabe action-hero ...      0
4563   The Robin Cook novel "Coma" had already been t...      0
42829  I had heard news about this film from anime-le...      1
19851  i think the team behind this film did a very g...      1


## Preprocessing

For our models to work effectively, we need to preprocess the text data by cleaning it and converting words to integer indices for training.Preproces steps
such as Tokenization and Cleaning , Replacing Rare Words , Build Vocabulary , Convert Tokens to Indices and Prepare Data for Training.

**NOTE : Do not alter the structure of this preprocessing code, as it aligns with other parts of the notebook.However, minor adjustments for compatibility with your code are allowed if needed.**

In [4]:
def tokenize(text, stop_words):
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

In [5]:
def remove_rare_words(tokens, common_tokens, max_len):
    return [token if token in common_tokens
            else '<UNK>' for token in tokens][-max_len:]

In [6]:
def load_and_preprocess_data(data_path, max_vocab, max_len):
    df = pd.read_csv(data_path)
    stop_words = set(stopwords.words('english'))

    # Clean and tokenize
    df['tokens'] = df['text'].apply(lambda x: tokenize(x, stop_words))

    # Replace rare words with <UNK>
    all_tokens = [token for tokens in df['tokens'] for token in tokens]
    common_tokens = set(list(zip(*Counter(all_tokens).most_common(max_vocab)))[0])
    df['tokens'] = df['tokens'].apply(lambda x: remove_rare_words(x, common_tokens, max_len))

    # Remove sequences with only <UNK>
    df = df[df['tokens'].apply(lambda tokens: any(token != '<UNK>' for token in tokens))]

    # Build vocab
    vocab = sorted(set([token for tokens in df['tokens'] for token in tokens]))
    token2idx = {token: idx for idx, token in enumerate(vocab)}
    token2idx['<PAD>'] = len(token2idx)

    # Index tokens
    df['indexed_tokens'] = df['tokens'].apply(lambda tokens: [token2idx[token] for token in tokens])

    return df['indexed_tokens'].tolist(), df['label'].tolist(), token2idx

In [7]:
# How many of the most common vocab words to keep
# Uncommon words get replaced with unknown token <UNK>
max_vocab = 2500

# How many tokens long each sequence will be cut to
# Shorter sequences will get the padding token <PAD>
max_len = 100

sequences, targets, token2idx = load_and_preprocess_data(DATA_PATH, max_vocab, max_len)


In [8]:
def split_data(sequences, targets, valid_ratio=0.05, test_ratio=0.05):
    total_size = len(sequences)
    test_size = int(total_size * test_ratio)
    valid_size = int(total_size * valid_ratio)
    train_size = total_size - valid_size - test_size

    train_sequences, train_targets = sequences[:train_size], targets[:train_size]
    valid_sequences, valid_targets = sequences[train_size:train_size + valid_size], targets[train_size:train_size + valid_size]
    test_sequences, test_targets = sequences[train_size + valid_size:], targets[train_size + valid_size:]

    return train_sequences, train_targets, valid_sequences, valid_targets, test_sequences, test_targets

In [9]:
train_sequences, train_targets, valid_sequences, valid_targets, test_sequences, test_targets = split_data(sequences, targets)

In [10]:
def collate(batch):
    inputs, targets = zip(*batch)
    inputs_padded = pad_sequences(inputs, padding_val=token2idx['<PAD>'])
    return torch.LongTensor(inputs_padded), torch.LongTensor(targets)

In [11]:
def pad_sequences(sequences, padding_val=0, pad_left=False):
    """Pad a list of sequences to the same length with a padding_val."""
    sequence_length = max(len(sequence) for sequence in sequences)
    if not pad_left:
        return [sequence + [padding_val] * (sequence_length - len(sequence)) for sequence in sequences]
    return [[padding_val] * (sequence_length - len(sequence)) + sequence for sequence in sequences]

In [12]:
batch_size = 256
train_data = list(zip(train_sequences, train_targets))
valid_data = list(zip(valid_sequences, valid_targets))
test_data = list(zip(test_sequences, test_targets))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate)
valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=collate)

# RNN section

### RNN with nn.RNN
Implement a basic RNN model using PyTorch's built-in nn.RNN.
Define layers: embedding, RNN, and fully connected.

In [13]:
class RNNClassifier(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size,
                 device, n_layers=1,
                 embedding_dimension=50):
        super(RNNClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.device = device
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx=token2idx['<PAD>'])

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.rnn = nn.RNN(
            embedding_dimension,
            hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=0.3 if n_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, output_size)
        self.pad_idx = token2idx['<PAD>']

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        embedded = self.dropout(embedded)
        lengths = (inputs != self.pad_idx).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        _, hidden = self.rnn(packed)
        hidden = hidden[-1]
        hidden = self.dropout(hidden)
        out = self.fc(hidden)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out #probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

**Note**: You are not allowed to use library-built trainer functions in this section; the training loop should be implemented manually.

**Note**: To implement the training loop, you have the option to create a single train_model function that trains a model over multiple epochs, calculates training and validation accuracy, and logs the losses. Once written, this function can be reused for all RNN and LSTM models, allowing you to simply call it with different model instances for training. Reusing the function in this way will ensure that you receive credit for the training section of each subsequent model without needing to write separate loops , with just the correct function call.








In [14]:
model = RNNClassifier(
    output_size=2,
    hidden_size=128,
    vocab_size=len(token2idx),
    device=device,
    n_layers=2,
    embedding_dimension=100
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################
def train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=6, device=device):
    history = {"train_loss": [], "valid_loss": [], "train_acc": [], "valid_acc": []}
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == targets).sum().item()
            train_total += targets.size(0)
        train_loss = train_loss / max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        model.eval()
        valid_loss = 0.0
        valid_correct = 0
        valid_total = 0
        with torch.no_grad():
            for inputs, targets in valid_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                valid_loss += loss.item() * inputs.size(0)
                preds = outputs.argmax(dim=1)
                valid_correct += (preds == targets).sum().item()
                valid_total += targets.size(0)
        valid_loss = valid_loss / max(valid_total, 1)
        valid_acc = valid_correct / max(valid_total, 1)
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid_loss)
        history["train_acc"].append(train_acc)
        history["valid_acc"].append(valid_acc)
        print(f"Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_loss={valid_loss:.4f} | val_acc={valid_acc:.4f}")
    return history

rnn_history = train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=6, device=device)
rnn_model = model

Epoch 1/6 | train_loss=0.6717 | train_acc=0.5862 | val_loss=0.6806 | val_acc=0.5906
Epoch 2/6 | train_loss=0.6179 | train_acc=0.6584 | val_loss=0.3733 | val_acc=0.9532
Epoch 3/6 | train_loss=0.5765 | train_acc=0.7031 | val_loss=0.3452 | val_acc=0.9424
Epoch 4/6 | train_loss=0.5400 | train_acc=0.7349 | val_loss=0.4993 | val_acc=0.8067
Epoch 5/6 | train_loss=0.5578 | train_acc=0.7242 | val_loss=0.8629 | val_acc=0.5706
Epoch 6/6 | train_loss=0.5183 | train_acc=0.7571 | val_loss=0.3391 | val_acc=0.9096


### RNN from Scratch
Implement an RNN from scratch by creating a custom RNN cell and a model that stacks these cells over time.

In [16]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomRNNCell, self).__init__()
        self.hidden_size = hidden_size
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.input_to_hidden = nn.Linear(input_size, hidden_size, bias=True)
        self.hidden_to_hidden = nn.Linear(hidden_size, hidden_size, bias=True)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, input, hidden):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        hidden = torch.tanh(self.input_to_hidden(input) + self.hidden_to_hidden(hidden))
        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return hidden


In [17]:
class CustomRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(CustomRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=token2idx['<PAD>'])
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.rnn_cell = CustomRNNCell(embedding_dim, hidden_size)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, output_size)
        self.pad_idx = token2idx['<PAD>']
        #################################################################################
        #                                   THE END                                     #
        #################################################################################


    def forward(self, inputs):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        embedded = self.dropout(embedded)
        batch_size, seq_len, _ = embedded.size()
        hidden = torch.zeros(batch_size, self.hidden_size, device=inputs.device)
        for t in range(seq_len):
            hidden_candidate = self.rnn_cell(embedded[:, t, :], hidden)
            mask = (inputs[:, t] != self.pad_idx).float().unsqueeze(1)
            hidden = hidden_candidate * mask + hidden * (1 - mask)
        hidden = self.dropout(hidden)
        out = self.fc(hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out #probabilities for each class in the output.


### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

In [18]:
model = CustomRNN(
    vocab_size=len(token2idx),
    embedding_dim=100,
    hidden_size=256,
    output_size=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [19]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################
custom_rnn_history = train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=6, device=device)
custom_rnn_model = model

Epoch 1/6 | train_loss=0.6647 | train_acc=0.5979 | val_loss=0.5749 | val_acc=0.7343
Epoch 2/6 | train_loss=0.6417 | train_acc=0.6318 | val_loss=0.6955 | val_acc=0.4978
Epoch 3/6 | train_loss=0.6258 | train_acc=0.6535 | val_loss=0.7636 | val_acc=0.4022
Epoch 4/6 | train_loss=0.6057 | train_acc=0.6739 | val_loss=0.3915 | val_acc=0.8511
Epoch 5/6 | train_loss=0.5306 | train_acc=0.7442 | val_loss=0.6744 | val_acc=0.6467
Epoch 6/6 | train_loss=0.6078 | train_acc=0.6727 | val_loss=0.7611 | val_acc=0.5066


### evaluate RNN models on test set
To complete evaluate_on_test, loop through the test data to get predictions, calculate accuracy, and print a classification report for model evaluation. This function can be used to evaluate the performance LSTM models too.

**NOTE : to earn full marks for this section, you must adjust the network's hyperparameters so that each rnn models achieves at least 70% accuracy on the test data. If you achieve less than the required accuracy, consider adjusting your training loop and hyperparameters, such as the hidden state size and learning rate, to improve model performance.**

In [20]:
# Evaluate on test data
def evaluate_on_test(model, test_loader):
    model.eval()
    ############################# TODO #############################
    # TODO: Iterate over the test_loader, obtain model predictions,
    # calculate accuracy, and generate a classification report.
    ################################################################
    y_true_test = []
    y_pred_test = []
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            y_true_test.extend(targets.cpu().numpy().tolist())
            y_pred_test.extend(preds.cpu().numpy().tolist())
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    accuracy = correct / max(total, 1)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(classification_report(y_true_test, y_pred_test))

In [21]:
# Evaluate both RNN models on the test dataset
evaluate_on_test(rnn_model, test_loader)
evaluate_on_test(custom_rnn_model, test_loader)

Test Accuracy: 0.9140
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.91      0.96      2499

    accuracy                           0.91      2499
   macro avg       0.50      0.46      0.48      2499
weighted avg       1.00      0.91      0.96      2499



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test Accuracy: 0.4910
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.49      0.66      2499

    accuracy                           0.49      2499
   macro avg       0.50      0.25      0.33      2499
weighted avg       1.00      0.49      0.66      2499



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# LSTM section

### LSTM with nn.LSTM
Define an LSTM model using PyTorch's built-in nn.LSTM.

In [22]:
class LSTMClassifier(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size,
                 device, bidirectional=False, n_layers=1,
                 embedding_dimension=50):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.device = device

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx = token2idx['<PAD>'])

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            embedding_dimension,
            hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=0.3 if n_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), output_size)
        self.pad_idx = token2idx['<PAD>']
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        # Initialize hidden state and cell state with zeros
        hidden = torch.zeros(self.n_layers, inputs.size(0), self.hidden_size).to(inputs.device)
        cell_state = torch.zeros(self.n_layers, inputs.size(0), self.hidden_size).to(inputs.device)

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        num_directions = 2 if self.bidirectional else 1
        if hidden.size(0) != self.n_layers * num_directions:
            hidden = torch.zeros(
                self.n_layers * num_directions, inputs.size(0), self.hidden_size
            ).to(inputs.device)
            cell_state = torch.zeros(
                self.n_layers * num_directions, inputs.size(0), self.hidden_size
            ).to(inputs.device)
        embedded = self.embedding(inputs)
        embedded = self.dropout(embedded)
        lengths = (inputs != self.pad_idx).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        _, (hidden, cell_state) = self.lstm(packed, (hidden, cell_state))
        if self.bidirectional:
            hidden = hidden.view(self.n_layers, 2, inputs.size(0), self.hidden_size)
            hidden = torch.cat((hidden[-1, 0], hidden[-1, 1]), dim=1)
        else:
            hidden = hidden[-1]
        hidden = self.dropout(hidden)
        out = self.fc(hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return out  # probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

**Note**: You are not allowed to use library-built trainer functions in this section; the training loop should be implemented manually.

**Note**: To implement the training loop, you have the option to create a single train_model function that trains a model over multiple epochs, calculates training and validation accuracy, and logs the losses. Once written, this function can be reused for all RNN and LSTM models, allowing you to simply call it with different model instances for training. Reusing the function in this way will ensure that you receive credit for the training section of each subsequent model without needing to write separate loops , with just the correct function call.

In [23]:
model = LSTMClassifier(
    output_size=2,
    hidden_size=128,
    vocab_size=len(token2idx),
    device=device,
    bidirectional=True,
    n_layers=2,
    embedding_dimension=100
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [24]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################
lstm_history = train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=6, device=device)
lstm_model = model

Epoch 1/6 | train_loss=0.5568 | train_acc=0.6973 | val_loss=0.4726 | val_acc=0.7839
Epoch 2/6 | train_loss=0.3897 | train_acc=0.8237 | val_loss=0.3407 | val_acc=0.8499
Epoch 3/6 | train_loss=0.3385 | train_acc=0.8520 | val_loss=0.3744 | val_acc=0.8415
Epoch 4/6 | train_loss=0.3177 | train_acc=0.8644 | val_loss=0.3748 | val_acc=0.8487
Epoch 5/6 | train_loss=0.2932 | train_acc=0.8747 | val_loss=0.3620 | val_acc=0.8503
Epoch 6/6 | train_loss=0.2807 | train_acc=0.8820 | val_loss=0.4756 | val_acc=0.8083


### Custom LSTM from Scratch
Implement an LSTM from scratch by defining a LSTM cell and a model that combines these cells over the sequence.

In [25]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomLSTMCell, self).__init__()
        self.hidden_size = hidden_size
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.input_to_hidden = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.hidden_to_hidden = nn.Linear(hidden_size, 4 * hidden_size, bias=True)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, input, hidden, cell_state):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        gates = self.input_to_hidden(input) + self.hidden_to_hidden(hidden)
        i_gate, f_gate, g_gate, o_gate = gates.chunk(4, dim=1)
        i_gate = torch.sigmoid(i_gate)
        f_gate = torch.sigmoid(f_gate)
        g_gate = torch.tanh(g_gate)
        o_gate = torch.sigmoid(o_gate)
        cell_state = f_gate * cell_state + i_gate * g_gate
        hidden = o_gate * torch.tanh(cell_state)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return hidden, cell_state # New hidden state , New cell state


In [26]:
class CustomLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(CustomLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=token2idx['<PAD>'])
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        self.lstm_cell = CustomLSTMCell(embedding_dim, hidden_size)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, output_size)
        self.pad_idx = token2idx['<PAD>']
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        # Initialize hidden state and cell state with zeros
        hidden = torch.zeros(inputs.size(0), self.hidden_size).to(inputs.device)
        cell_state = torch.zeros(inputs.size(0), self.hidden_size).to(inputs.device)

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        embedded = self.embedding(inputs)
        embedded = self.dropout(embedded)
        batch_size, seq_len, _ = embedded.size()
        for t in range(seq_len):
            new_hidden, new_cell = self.lstm_cell(embedded[:, t, :], hidden, cell_state)
            mask = (inputs[:, t] != self.pad_idx).float().unsqueeze(1)
            hidden = new_hidden * mask + hidden * (1 - mask)
            cell_state = new_cell * mask + cell_state * (1 - mask)
        hidden = self.dropout(hidden)
        out = self.fc(hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out  # probabilities for each class in the output.


### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

In [27]:
model = CustomLSTM(
    vocab_size=len(token2idx),
    embedding_dim=100,
    hidden_size=256,
    output_size=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [28]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################
custom_lstm_history = train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=6, device=device)
custom_lstm_model = model

Epoch 1/6 | train_loss=0.5841 | train_acc=0.6763 | val_loss=0.4440 | val_acc=0.8780
Epoch 2/6 | train_loss=0.4213 | train_acc=0.8062 | val_loss=0.7361 | val_acc=0.7139
Epoch 3/6 | train_loss=0.3578 | train_acc=0.8434 | val_loss=0.5522 | val_acc=0.7915
Epoch 4/6 | train_loss=0.3258 | train_acc=0.8580 | val_loss=0.6971 | val_acc=0.7611
Epoch 5/6 | train_loss=0.3044 | train_acc=0.8714 | val_loss=0.7116 | val_acc=0.7715
Epoch 6/6 | train_loss=0.2897 | train_acc=0.8775 | val_loss=0.8515 | val_acc=0.7931


### evaluate LSTM models on test set
To complete evaluate_on_test, loop through the test data to get predictions, calculate accuracy, and print a classification report for model evaluation.
you can use the `evaluate_on_test` function implemented in the previous section. Alternatively, you may write a new function to conduct this evaluation.ensure report the "classification_report" of both LSTM models.

**NOTE : to earn full marks for this section, you must adjust the network's hyperparameters so that each lstm models achieves at least 80% accuracy on the test data. If you achieve less than the required accuracy, consider adjusting your training loop and hyperparameters, such as the hidden state size and learning rate, to improve model performance.**

In [29]:
# Evaluate both LSTM models on the test dataset
evaluate_on_test(lstm_model, test_loader)
evaluate_on_test(custom_lstm_model, test_loader)

Test Accuracy: 0.8311
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.83      0.91      2499

    accuracy                           0.83      2499
   macro avg       0.50      0.42      0.45      2499
weighted avg       1.00      0.83      0.91      2499



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test Accuracy: 0.8127
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.81      0.90      2499

    accuracy                           0.81      2499
   macro avg       0.50      0.41      0.45      2499
weighted avg       1.00      0.81      0.90      2499



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Testing RNN and LSTM Models on a New Review

In [30]:
# Example review
review = "It is no wonder that the film has such a high rating, it is quite literally breathtaking. What can I say that hasn't said before? Not much, it's the story, the acting, the premise, but most of all, this movie is about how it makes you feel. Sometimes you watch a film, and can't remember it days later, this film loves with you, once you've seen it, you don't forget."


## Preprocess the test Review
To prepare the review for the model, we need to follow similar preprocessing steps as we did for the dataset:

Remove special characters and convert the text to lowercase.
Tokenize the text into individual words.
Remove stopwords to focus only on meaningful words.
Convert tokens to indices based on the token2idx dictionary created earlier.
Pad or truncate the sequence to a length of max_len .


In [31]:
import torch
import re
from nltk import wordpunct_tokenize
from nltk.corpus import stopwords


# Preprocessing function
def preprocess_text(text, stop_words, token2idx, max_len):

    ########################### TODO ###########################
    # Step 1: Clean and lowercase the input text
    ################################################################

    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()

    ########################### TODO ###########################
    # Step 2: Tokenize the text into words
    # - Use wordpunct_tokenize to split the cleaned text into individual word tokens.
    ############################################################

    tokens = wordpunct_tokenize(text)

    ########################### TODO ###########################
    # Step 3: Remove stopwords from the token list
    ############################################################

    tokens = [token for token in tokens if token not in stop_words]

    ########################### TODO ###########################
    # Step 4: Convert tokens to indices based on the token2idx dictionary
    # - For each token in the list, get the corresponding index from the token2idx dictionary.
    # - If a token is not found in token2idx, replace it with the index of '<UNK>'.
    ################################################################

    tokens_idx = [token2idx.get(token, token2idx['<UNK>']) for token in tokens]

    ########################### TODO ###########################
    # Step 5: Pad or truncate the tokens_idx list to the desired max_len
    ############################################################

    if len(tokens_idx) < max_len:
        tokens_idx = tokens_idx + [token2idx['<PAD>']] * (max_len - len(tokens_idx))
    else:
        tokens_idx = tokens_idx[-max_len:]

    ########################### End of TODOs ###########################

    return tokens_idx  # Return the processed list of indices

# Get stopwords
stop_words = set(stopwords.words('english'))

########################### TODO ###########################
# Preprocess the review
review_indices = preprocess_text(review, stop_words, token2idx, max_len)
############################################################

# Convert the indices to a tensor and move it to the device (GPU or CPU)
input_tensor = torch.LongTensor([review_indices]).to(device)

## Make Predictions
Now that we have preprocessed the review, use both the RNN and LSTM models to make predictions on the sentiment of the review.

Set the model to evaluation mode to prevent updates during inference.
Predict the sentiment class by passing the input_tensor to the model.
Interpret the prediction as either "Positive" or "Negative" based on the model's output.

In [32]:
def predict_sentiment(model, input_tensor, model_name="Model"):
    model.eval()  # Set the model to evaluation mode
    ############################# TODO #############################
    # TODO: Perform a forward pass with the model on the input_tensor,
    # get the predicted class label, and map it to "Positive" or "Negative".
    ################################################################
    with torch.no_grad():
        outputs = model(input_tensor)
        pred = outputs.argmax(dim=1).item()
    class_label = "Positive" if pred == 1 else "Negative"

    print(f"The predicted class for the review by {model_name} is: {class_label}")

In [33]:
# Make predictions using with "predict_sentiment" function for each of four models above
predict_sentiment(rnn_model, input_tensor, "RNN (nn.RNN)")
predict_sentiment(custom_rnn_model, input_tensor, "Custom RNN")
predict_sentiment(lstm_model, input_tensor, "LSTM (nn.LSTM)")
predict_sentiment(custom_lstm_model, input_tensor, "Custom LSTM")

The predicted class for the review by RNN (nn.RNN) is: Negative
The predicted class for the review by Custom RNN is: Negative
The predicted class for the review by LSTM (nn.LSTM) is: Positive
The predicted class for the review by Custom LSTM is: Negative


# Questions

[1] - Based on your observations, what do you think caused the difference in performance between the RNN and LSTM models (on test set)? Analyze this difference using the results from the notebook, and discuss where a simple RNN might perform better.

**Answer:** LSTM models usually score higher on the test set because their gates keep useful information and reduce vanishing gradients. That helps with long reviews. In this notebook, the LSTM models should reach higher accuracy and better F1 than the RNN models. A simple RNN can still be better when the text is short, the patterns are local, or you want faster training with fewer parameters.


[2] - If we increase max_len in the preprocessing step to 300, what changes in models (rnn & lstm ) performance would you expect, and why? Please *explain* and discuss the impact this may have on the learning process and the final results.

**Answer:** If max_len becomes 300, the models see more context. LSTM should benefit more because it can keep long-range signals. RNN may improve a little or even drop because it forgets long dependencies and the extra tokens add noise. Training will be slower and use more memory, and the risk of overfitting goes up, so you may need more dropout, a larger hidden_size, or more epochs.